# Scan2Stage — M3 Room Geometry Normalization
FBX → meters → Z-up → floor Z=0 → wall planes → 2D footprint.


In [ ]:
REPO_URL='https://github.com/6564200/Scan2Stage.git'
REPO_REF='milestone/room-normalization-colab'
WORKDIR='/content/Scan2Stage'
!rm -rf {WORKDIR}
!git clone -b {REPO_REF} {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!bash scripts/colab_bootstrap.sh


## Upload textured FBX
For the current Polycam example the source coordinates are centimeters, so `UNIT_SCALE=0.01`.


In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()
name=next(iter(uploaded))
assert name.lower().endswith('.fbx'), 'Upload an FBX file'
INPUT=Path('/content/Scan2Stage/data')/Path(name).name
INPUT.parent.mkdir(parents=True, exist_ok=True)
Path(name).replace(INPUT)
print(INPUT)


In [ ]:
OUT=Path('/content/Scan2Stage/outputs/m3_room')
UNIT_SCALE=0.01
!scan2stage {INPUT} --output-dir {OUT} --samples 300000 --source-up y --unit-scale {UNIT_SCALE}


In [ ]:
import json
report=json.loads((OUT/'report.json').read_text())
room=json.loads((OUT/'room_geometry.json').read_text())
print('Bounds in meters:', report['bounds_meters']['extent'])
print('Floor before translation, m:', room['floor']['z_m'])
print('Estimated room height, m:', room['estimated_room_height_m'])
print('Detected walls:', len(room['walls']))
print('Footprint vertices:', len(room['footprint_xy_m']))
room


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fp=np.asarray(room['footprint_xy_m'])
if len(fp):
    closed=np.vstack([fp, fp[0]])
    plt.figure(figsize=(8,8))
    plt.plot(closed[:,0], closed[:,1], '-o')
    plt.axis('equal')
    plt.xlabel('X, m'); plt.ylabel('Y, m'); plt.title('Estimated room footprint')
    plt.grid(True)


In [ ]:
!pytest -q
